In [ ]:
import os
import warnings
import torch
import numpy as np
import tensorflow as tf
from PIL import Image
from transformers import (
    Qwen2VLForConditionalGeneration,
    AutoProcessor,
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TFBertModel
)
from qwen_vl_utils import process_vision_info

# CONFIGURACIÓN DE ENTORNO
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_USE_LEGACY_KERAS"] = "1"
warnings.filterwarnings("ignore")

# TensorFlow en CPU para evitar conflictos de VRAM con Qwen (GPU)
tf.config.set_visible_devices([], "GPU")

# RUTAS Y CARGA
RUTA_MODELO_KERAS = "modelo_MEMES4GOOD_v3.keras" 
MODEL_BERT_NAME = "dccuchile/bert-base-spanish-wwm-cased"

# 1. Modelos de Visión y Lenguaje (GPU)
model_qwen = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct", torch_dtype="auto", device_map="cuda"
)
processor_qwen = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

# 2. Traductor (GPU)
translator_tokenizer = AutoTokenizer.from_pretrained("Helsinki-NLP/opus-mt-en-es")
translator_model = AutoModelForSeq2SeqLM.from_pretrained("Helsinki-NLP/opus-mt-en-es").to("cuda")

# 3. Clasificador Multimodal (CPU)
tokenizer_bert = AutoTokenizer.from_pretrained(MODEL_BERT_NAME)
model_juez = tf.keras.models.load_model(
    RUTA_MODELO_KERAS,
    custom_objects={"TFBertModel": TFBertModel},
    compile=False
)

# FUNCIONES DE PROCESAMIENTO

def _traducir(text):
    if not text or not text.strip(): return ""
    inputs = translator_tokenizer(text, return_tensors="pt", padding=True).to("cuda")
    out = translator_model.generate(**inputs, max_new_tokens=200)
    return translator_tokenizer.batch_decode(out, skip_special_tokens=True)[0]

def _analizar_qwen(image):
    # Prompt 1: Extracción de texto (OCR) - Se queda igual
    msg_ocr = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": "Extract all visible text. Return only the text."}]}]
    text_ocr = processor_qwen.apply_chat_template(msg_ocr, tokenize=False, add_generation_prompt=True)
    inputs_ocr = processor_qwen(text=[text_ocr], images=process_vision_info(msg_ocr)[0], return_tensors="pt", padding=True).to("cuda")
    ids_ocr = model_qwen.generate(**inputs_ocr, max_new_tokens=100)
    ocr = processor_qwen.batch_decode(ids_ocr[:, inputs_ocr.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()

    # Prompt 2: DESCRIPCIÓN 
    msg_desc = [{"role": "user", "content": [
        {"type": "image", "image": image}, 
        {"type": "text", "text": "Describe visual context, characters, their gestures and the relationship between elements. Focus on potential stereotypes or controversial symbols. One dense sentence. No transcription. No filler. No translation. Style: Telegraphic."}
    ]}]
    text_desc = processor_qwen.apply_chat_template(msg_desc, tokenize=False, add_generation_prompt=True)
    inputs_desc = processor_qwen(text=[text_desc], images=process_vision_info(msg_desc)[0], return_tensors="pt", padding=True).to("cuda")
    ids_desc = model_qwen.generate(**inputs_desc, max_new_tokens=150)
    desc_en = processor_qwen.batch_decode(ids_desc[:, inputs_desc.input_ids.shape[1]:], skip_special_tokens=True)[0].strip()
    
    return ocr, _traducir(desc_en)

def _predecir(image, ocr, desc):
    # Imagen: EfficientNetB0
    img_resized = image.resize((224, 224))
    img_array = tf.keras.preprocessing.image.img_to_array(img_resized)
    img_batch = np.expand_dims(tf.keras.applications.efficientnet.preprocess_input(img_array), axis=0)

    # Texto: Formato [SEP] y MAX_LEN 160
    txt_fusion = f"MEME: {ocr} [SEP] ESCENA: {desc}"
    tokens = tokenizer_bert(
        [txt_fusion], max_length=160, padding="max_length", truncation=True, return_tensors="tf"
    )

    # Inferencia
    prediction = model_juez.predict(
        [img_batch, tokens["input_ids"], tokens["attention_mask"]], verbose=0
    )[0][0]
    return float(prediction)

In [ ]:
# --- INTERFAZ GRADIO ---
import gradio as gr

def inference_pipeline(input_img):
    if input_img is None: return "Error: No se ha subido ninguna imagen."
    
    ocr, desc_es = _analizar_qwen(input_img)
    score = _predecir(input_img, ocr, desc_es)
    
    # 0 = Ofensivo, 1 = Inofensivo 
    es_ofensivo = score < 0.5
    confianza = (1 - score) if es_ofensivo else score
    label = "DAÑINO / OFENSIVO" if es_ofensivo else "INOFENSIVO / SEGURO"
    
    return {
        "Veredicto": label,
        "Confianza": f"{confianza:.2%}",
        
    }

demo = gr.Interface(
    fn=inference_pipeline,
    inputs=gr.Image(type="pil", label="Sube el meme"),
    outputs=gr.JSON(label="Análisis Multimodal"),
    title="Detector Multimodal de Memes Ofensivos V3",
)

demo.launch()